In [8]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, random_split
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image
import kagglehub
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg') 
import matplotlib.pyplot as plt

In [9]:
# ==========================================
# 1. 全局配置与路径 (Configuration)
# ==========================================
print(">>> Step 1: Configuration...")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {DEVICE}")

# 超参数
BATCH_SIZE = 128
IMG_SIZE = 64
LATENT_DIM = 100
EPOCHS = 50
LEARNING_RATE = 1e-3
BETA = 1.0       # Gaussian KL Weight
MMD_LAMBDA = 100.0 # Uniform MMD Weight

# 路径配置
BASE_DIR = "vae_experiment_results"
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
GEN_IMG_DIR = os.path.join(BASE_DIR, "generated_images")
LOG_DIR = os.path.join(BASE_DIR, "logs")

for d in [CHECKPOINT_DIR, GEN_IMG_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

>>> Step 1: Configuration...
Using Device: cuda


In [10]:
# ==========================================
# 2. 数据加载 (Data Loading - CPU Safe Mode)
# ==========================================
print("\n>>> Step 2: Loading Data...")

dataset_path = kagglehub.dataset_download("splcher/animefacedataset")
image_folder_path = os.path.join(dataset_path, 'images')

transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

image_files = [os.path.join(image_folder_path, f) for f in os.listdir(image_folder_path) if f.endswith(('.png', '.jpg'))]
tensor_list = []

# 使用 tqdm 读取，增加错误处理
for img_path in tqdm(image_files, desc="Reading Images"):
    try:
        img = Image.open(img_path).convert("RGB")
        tensor_list.append(transform(img))
    except:
        continue

full_dataset_cpu = torch.stack(tensor_list) 
total_count = len(full_dataset_cpu)
print(f"Total Images: {total_count}")

# 划分数据集
train_size = int(0.8 * total_count)
val_size = int(0.1 * total_count)
test_size = total_count - train_size - val_size

full_dataset = TensorDataset(full_dataset_cpu)
train_set, val_set, test_set = random_split(full_dataset, [train_size, val_size, test_size])

# num_workers=0 保证最强兼容性
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


>>> Step 2: Loading Data...


Reading Images: 100%|██████████| 63565/63565 [00:28<00:00, 2265.54it/s]


Total Images: 63565


In [11]:
# ==========================================
# 3. 模型定义 (Models)
# ==========================================
class VAE(nn.Module):
    def __init__(self, latent_dim=100):
        super(VAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(64, 128, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(128, 256, 4, 2, 1), nn.ReLU(),
            nn.Flatten()
        )
        self.fc_mu = nn.Linear(256 * 4 * 4, latent_dim)
        self.fc_log_var = nn.Linear(256 * 4 * 4, latent_dim)
        self.decoder_fc = nn.Linear(latent_dim, 256 * 4 * 4)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Tanh()
        )
    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_log_var(h)
    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std
    def decode(self, z):
        h = self.decoder_fc(z).view(-1, 256, 4, 4)
        return self.decoder(h)
    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        return self.decode(z), mu, log_var

class UniformVAE(nn.Module):
    def __init__(self, latent_dim=100):
        super(UniformVAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(64, 128, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(128, 256, 4, 2, 1), nn.ReLU(),
            nn.Flatten()
        )
        self.fc_center = nn.Linear(256 * 4 * 4, latent_dim)
        self.fc_width = nn.Linear(256 * 4 * 4, latent_dim) 
        self.decoder_fc = nn.Linear(latent_dim, 256 * 4 * 4)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Tanh() 
        )
    def encode(self, x):
        h = self.encoder(x)
        center = self.fc_center(h)
        width = F.softplus(self.fc_width(h)) + 1e-4 
        return center, width
    def reparameterize(self, center, width):
        if self.training:
            noise = torch.rand_like(center) * 2 - 1 
            return center + width * noise
        else:
            return center
    def decode(self, z):
        h = self.decoder_fc(z).view(-1, 256, 4, 4)
        return self.decoder(h)
    def forward(self, x):
        center, width = self.encode(x)
        z = self.reparameterize(center, width)
        return self.decode(z), z


In [12]:
# ==========================================
# 4. Loss Functions
# ==========================================
def loss_function_gaussian(recon_x, x, mu, log_var):
    recon_loss = F.mse_loss(recon_x, x, reduction='sum')
    kld = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    return recon_loss + BETA * kld

def compute_kernel(x, y):
    x_size, dim = x.shape[0], x.shape[1]
    y_size = y.shape[0]
    tiled_x = x.view(x_size, 1, dim).repeat(1, y_size, 1)
    tiled_y = y.view(1, y_size, dim).repeat(x_size, 1, 1)
    return torch.exp(-(1.0/dim) * ((tiled_x - tiled_y)**2).sum(dim=2))

def compute_mmd(x, y):
    x_kernel = compute_kernel(x, x)
    y_kernel = compute_kernel(y, y)
    xy_kernel = compute_kernel(x, y)
    return x_kernel.mean() + y_kernel.mean() - 2 * xy_kernel.mean()

def loss_function_uniform(recon_x, x, z, latent_dim, device):
    recon_loss = F.mse_loss(recon_x, x, reduction='sum')
    true_samples = (torch.rand(z.shape[0], latent_dim) * 2 - 1).to(device)
    mmd_loss = compute_mmd(z, true_samples)
    return recon_loss + MMD_LAMBDA * mmd_loss


In [13]:
# ==========================================
# 5. 训练引擎
# ==========================================
def run_training(model_name, model, optimizer, loss_fn, train_loader, val_loader):
    print(f"\n[Info] Starting Training: {model_name} VAE")
    history = {"train_loss": [], "val_recon_mse": []}
    
    # Checkpoint logic
    latest_ckpt = os.path.join(CHECKPOINT_DIR, f"{model_name}_latest.pth")
    start_epoch = 0
    if os.path.exists(latest_ckpt):
        print(f"Loading checkpoint from {latest_ckpt}")
        try:
            ckpt = torch.load(latest_ckpt)
            model.load_state_dict(ckpt['model'])
            optimizer.load_state_dict(ckpt['optim'])
            start_epoch = ckpt['epoch']
        except Exception as e:
            print(f"Failed to load checkpoint, starting fresh. Error: {e}")

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        train_loss_sum = 0.0
        
        pbar = tqdm(train_loader, desc=f"[{model_name}] Ep {epoch+1}/{EPOCHS}", leave=False)
        for (batch_imgs,) in pbar:
            batch_imgs = batch_imgs.to(DEVICE)
            optimizer.zero_grad()
            
            if model_name == "Gaussian":
                recon, mu, log_var = model(batch_imgs)
                loss = loss_fn(recon, batch_imgs, mu, log_var)
            else:
                recon, z = model(batch_imgs)
                loss = loss_fn(recon, batch_imgs, z, LATENT_DIM, DEVICE)
            
            loss.backward()
            optimizer.step()
            train_loss_sum += loss.item()
            
        avg_train_loss = train_loss_sum / len(train_loader.dataset)
        history["train_loss"].append(avg_train_loss)
        
        # Validation
        model.eval()
        val_mse_sum = 0.0
        with torch.no_grad():
            for (batch_imgs,) in val_loader:
                batch_imgs = batch_imgs.to(DEVICE)
                if model_name == "Gaussian":
                    recon, _, _ = model(batch_imgs)
                else:
                    recon, _ = model(batch_imgs)
                mse = F.mse_loss(recon, batch_imgs, reduction='sum')
                val_mse_sum += mse.item()
                
        avg_val_mse = val_mse_sum / len(val_loader.dataset)
        history["val_recon_mse"].append(avg_val_mse)
        
        print(f"    Epoch {epoch+1} | Loss: {avg_train_loss:.4f} | Val MSE: {avg_val_mse:.4f}")

        # Save Checkpoint
        torch.save({
            'epoch': epoch + 1, 'model': model.state_dict(), 'optim': optimizer.state_dict()
        }, latest_ckpt)

        # Save Images
        if (epoch + 1) % 5 == 0 or (epoch + 1) == EPOCHS:
            with torch.no_grad():
                if model_name == "Gaussian":
                    z = torch.randn(64, LATENT_DIM).to(DEVICE)
                else:
                    z = (torch.rand(64, LATENT_DIM) * 2 - 1).to(DEVICE)
                gen_imgs = model.decode(z).cpu()
                save_image(gen_imgs, os.path.join(GEN_IMG_DIR, f"{model_name}_epoch_{epoch+1}.png"), normalize=True, nrow=8)

    return history

In [14]:
# ==========================================
# 6. 执行训练
# ==========================================
print("\n>>> Step 6: Running Experiments...")

print("\n--- Experiment 1: Gaussian VAE ---")
vae_g = VAE(LATENT_DIM).to(DEVICE)
opt_g = optim.Adam(vae_g.parameters(), lr=LEARNING_RATE)
hist_g = run_training("Gaussian", vae_g, opt_g, loss_function_gaussian, train_loader, val_loader)

print("\n--- Experiment 2: Uniform VAE ---")
vae_u = UniformVAE(LATENT_DIM).to(DEVICE)
opt_u = optim.Adam(vae_u.parameters(), lr=LEARNING_RATE)
hist_u = run_training("Uniform", vae_u, opt_u, loss_function_uniform, train_loader, val_loader)


>>> Step 6: Running Experiments...

--- Experiment 1: Gaussian VAE ---

[Info] Starting Training: Gaussian VAE
Loading checkpoint from vae_experiment_results\checkpoints\Gaussian_latest.pth

--- Experiment 2: Uniform VAE ---

[Info] Starting Training: Uniform VAE
Loading checkpoint from vae_experiment_results\checkpoints\Uniform_latest.pth


In [16]:
import glob
from torchvision.utils import make_grid

# --- 2. 准备数据 (只加载前 32 张做测试) ---
print("Step 2: Loading a small batch for visualization...")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 64
LATENT_DIM = 100
BATCH_SIZE = 32

transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)), # 3 channels
])

# 搜索所有图片文件
image_files = []
for ext in ['*.png', '*.jpg', '*.jpeg', '*.PNG']:
    image_files.extend(glob.glob(os.path.join(image_folder_path, ext)))

# 只取前 32 张
image_files = sorted(image_files)[:32]

if len(image_files) == 0:
    raise RuntimeError(f"No images found in {image_folder_path}. Please check the path.")

tensors = []
for f in image_files:
    try:
        img = Image.open(f).convert("RGB")
        tensors.append(transform(img))
    except Exception as e:
        print(f"Skipping bad file {f}: {e}")

# 堆叠成 Batch
vis_batch = torch.stack(tensors).to(DEVICE)
print(f"Loaded batch shape: {vis_batch.shape}")

# --- 4. 加载 Checkpoint 并推理 ---
print("Step 3: Loading models and inferencing...")
CHECKPOINT_DIR = "vae_experiment_results/checkpoints"

vae_g = VAE(LATENT_DIM).to(DEVICE)
vae_u = UniformVAE(LATENT_DIM).to(DEVICE)

# 容错加载: 检查 key 是否包含 'model_state_dict' 或直接是 dict
def safe_load(model, path):
    if not os.path.exists(path):
        print(f"Warning: Checkpoint not found at {path}")
        return
    ckpt = torch.load(path, map_location=DEVICE)
    state_dict = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt.get('model', ckpt)
    model.load_state_dict(state_dict)
    model.eval()
    print(f"Loaded {path}")

safe_load(vae_g, os.path.join(CHECKPOINT_DIR, "Gaussian_latest.pth"))
safe_load(vae_u, os.path.join(CHECKPOINT_DIR, "Uniform_latest.pth"))

with torch.no_grad():
    recon_g, _, _ = vae_g(vis_batch)
    recon_u, _ = vae_u(vis_batch)

    # 计算这一个小Batch的MSE
    mse_g = F.mse_loss(recon_g, vis_batch, reduction='sum') / 32
    mse_u = F.mse_loss(recon_u, vis_batch, reduction='sum') / 32

print(f"\nMini-Batch MSE Comparison:")
print(f"Gaussian: {mse_g:.4f}")
print(f"Uniform:  {mse_u:.4f}")

# --- 5. 画图 (只画前8张) ---
print("Step 4: Generating comparison plot...")
n_show = 8
# 拼接：原图 | Gaussian重构 | Uniform重构
comparison = torch.cat([
    vis_batch[:n_show], 
    recon_g[:n_show], 
    recon_u[:n_show]
])

# 反归一化 [-1,1] -> [0,1]
comparison = comparison.cpu() * 0.5 + 0.5
grid = make_grid(comparison, nrow=n_show, padding=2)

plt.figure(figsize=(16, 6))
plt.imshow(grid.permute(1, 2, 0).clip(0, 1))
plt.axis('off')
plt.title(f"Row 1: Original | Row 2: Gaussian VAE (MSE {mse_g:.1f}) | Row 3: Uniform VAE (MSE {mse_u:.1f})")
plt.tight_layout()
plt.show()

Step 2: Loading a small batch for visualization...
Loaded batch shape: torch.Size([32, 3, 64, 64])
Step 3: Loading models and inferencing...
Loaded vae_experiment_results/checkpoints\Gaussian_latest.pth
Loaded vae_experiment_results/checkpoints\Uniform_latest.pth

Mini-Batch MSE Comparison:
Gaussian: 603.9198
Uniform:  534.2500
Step 4: Generating comparison plot...


C:\Users\yangc\AppData\Local\Temp\ipykernel_9684\3571218681.py:93: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
